In [ ]:
options(future.globals.maxSize = 100000000 * 1024^2)

# set random seed for reproducibility
set.seed(12345)

# ========================= Libraries ===================================


## We load the required packages
library(dplyr)
library(Seurat)
library(readr)
library(readr)
# single-cell analysis package
# plotting and data science packages
# library(tidyverse)
library(cowplot)
library(patchwork)
library(ggplot2)
# co-expression network analysis packages:
library(igraph)
library(harmony)
library(dittoSeq)
library(RColorBrewer)

library(org.Hs.eg.db)
require(tidyverse)
library(clusterProfiler)
library(WGCNA)
library(hdWGCNA)
library(scCustomize)
library(harmony)


library(SCopeLoomR)
library(dreamlet)
library(SingleCellExperiment)
library(scater)
# library(UCell)
# library(Nebulosa)

library(SingleCellExperiment)
library(scuttle)
library(Seurat)
library(doParallel)
'%notin%' <- Negate('%in%')

# plotting
library(ggplot2)
library(dplyr)
library(RColorBrewer)
library(cowplot)
library(ggtree)
library(aplot)
library(circlize)
library(ComplexHeatmap)

# meta
library(tidyr)
library(muscat)
library(broom)
library(tidyverse)
library(metafor) 

input_rds <- args[1]
group_by <- args[2]
only_microglia <- as.logical(args[3])
n_of_aggregated <- as.numeric(args[4]) # 25
n_of_aggregated_text <- args[4]
threads <- as.numeric(args[5])
output_pdf <- args[6]

workdir <- "7.WGCNA/"
source('utils.R')
dir.create(workdir)
registerDoParallel(cores=threads)
colors <- brewer.pal(n = 9, name = 'Set3')
colors[2] <- colors[8]
colors[8] <- "azure3"
colors_rb <- rev(brewer.pal(n = 6, name = 'RdGy'))


# Read Metacells rds

In [2]:
WGCNA_dir = '7.WGCNA/'
input_rds <-  paste0(WGCNA_dir,'2024_03_28_Myeloid_Metacells_Subclass_ADAM.rds.ztsd')

Metacells <- readCRDS(input_rds)
Metacells$TwoClusters <-  as.vector(Metacells$Clusters)
Metacells$TwoClusters[Metacells$Clusters %in% c("MTC_1","MTC_2","MTC_3")] <- "MTC_123"
Metacells$TwoClusters[Metacells$Clusters %in% c("MTC_4","MTC_5","MTC_6")]<- "MTC_456"


# Subset only the two main metaclusters

In [3]:
subset_two_cl <- subset(Metacells,TwoClusters%in% c('MTC_123','MTC_456'))

# Find DEGs between the two main metaclusters

In [4]:
# Find all markers using the FindAllMarkers function, considering only positive markers
Idents(subset_two_cl) <- "TwoClusters"
DEGs <- FindAllMarkers(subset_two_cl,only.pos = T)


Calculating cluster MTC_456

Calculating cluster MTC_123



# Filter the one with log2FC > 0.25

In [5]:
DEGs %>%
    group_by(cluster) %>%
    dplyr::filter(avg_log2FC > 0.25) %>% arrange(desc(avg_log2FC)) %>%
    ungroup() -> top15


# Calculate Average expression for the heatmap

In [6]:
AverageExpression_RNA <- AverageExpression(object = subset_two_cl, group.by = c('Clusters', 'TwoClusters'),features =top15$gene,return.seurat = T )


Warning message:
"`invoke()` is deprecated as of rlang 0.4.0.
Please use `exec()` or `inject()` instead.
This warning is displayed once every 8 hours."
Centering and scaling data matrix



In [7]:
label <- as.vector(colnames(AverageExpression_RNA))
# Split each string based on underscore "_"
split_strings <- strsplit(label, "_")

# Extract values before and after the underscore
before_underscore <- sapply(split_strings, function(x) x[1])
after_underscore <- sapply(split_strings, function(x) x[2])


In [8]:
options(repr.plot.width=7, repr.plot.height=8)
# Example: Define metadata columns for grouping
group_by_metadata_1 <- "TwoClusters"
group_by_metadata_2 <- "Clusters"

# Example: Define selected gene list
selected_genes <- top15$gene

# Extract metadata columns for grouping
metadata_1 <- before_underscore
metadata_2 <- after_underscore
# Create a data frame for metadata
metadata_df <- data.frame(Clusters = metadata_1, TClusters = metadata_2)


# Extract expression matrix
gene_expression_matrix <- AverageExpression_RNA@assays$RNA@scale.data
colnames(gene_expression_matrix) <- metadata_1
# Filter expression matrix for selected genes
selected_genes_expression <- as.matrix(gene_expression_matrix[selected_genes, , drop = FALSE])


# Group the expression matrix by metadata columns
grouped_data <- split(selected_genes_expression, metadata_df)

ggplotColours <- function(n = 6, h = c(0, 360) + 15){
  if ((diff(h) %% 360) < 1) h[2] <- h[2] - 360/n
  hcl(h = (seq(h[1], h[2], length = n)), c = 100, l = 65)
}

color_list <- ggplotColours(n=9)

top_annotation <- HeatmapAnnotation(Clusters = metadata_1,TClusters = metadata_2,
                                   col=list(
                                       TClusters= c('MTC-456' = colors_rb[c(6)] ,'MTC-123'=colors_rb[c(2)]),
                                       Clusters=c('MTC-1'=color_list[1],'MTC-2'=color_list[2],
                                                  'MTC-3'=color_list[3],
                                                 'MTC-4'=color_list[4],
                                                  'MTC-5'=color_list[5],
                                                  'MTC-6'=color_list[6])
                                   ))

degs_with_interleukin <- rownames(selected_genes_expression)[grep('^IL',rownames(selected_genes_expression))]
selected_genes_plot_list <- c('IL3RA','APOE','SPP1','KLF2','SORL1','FOSL2','FOSL1','CD163','MAFB',
                              'PPARG','TCF4','BHLHE41',
                             'FOS','JUN','B2M','TREM2','CLU',
                            'CX3CR1','IL1B','MITF','PTPRG','DPYD','ITGA4','ITGB1','LILRB1','LGALS1','CD63','CD81','CD9')#'PGK1', 
#'S100A8','S100A9',
selected_genes_plot <- selected_genes_plot_list[selected_genes_plot_list%in% rownames(selected_genes_expression)]
index_genes <- rownames(selected_genes_expression) %in% selected_genes_plot
selected_genes_plot_ord <- rownames(selected_genes_expression)[index_genes]
ht = Heatmap(selected_genes_expression, 
    
    name = "z-score",
#     cluster_rows = FALSE,
    row_km = 3,
    col = colorRamp2(c(-2, 0, 2), c("blue", "white", "red")),
    
    top_annotation = top_annotation,
             use_raster=F,
             show_row_names = T)+

    rowAnnotation(link = anno_mark(at = which(index_genes), 
        labels = selected_genes_plot_ord, 
        labels_gp = gpar(fontsize = 14), padding = unit(1, "mm")))


In [9]:
pdf('Figure2_D.pdf',width = 8,height=8)
set.seed(88888)
options(repr.plot.width=8, repr.plot.height=8)
ht
dev.off()


png 
  2

# Save DEGs / Use them as input for IREA

In [13]:
# Order the markers in deg_pa dataframe by average log2 fold change in decreasing order
DEGs_ord <- DEGs[order(DEGs$avg_log2FC,decreasing = T),]
head(DEGs_ord)


,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
SPP1,0,3.610863,1.000,1.000,0,MTC_456,SPP1
TIMP1,0,3.235379,0.991,0.861,0,MTC_456,TIMP1
MT2A,0,3.219268,0.991,0.851,0,MTC_456,MT2A
LGALS1,0,3.157293,0.989,0.763,0,MTC_456,LGALS1
G0S2,0,3.022311,0.796,0.140,0,MTC_456,G0S2
CTSL,0,2.681548,0.999,0.989,0,MTC_456,CTSL


In [11]:
for_IREA <- DEGs_ord %>% filter(avg_log2FC >1, p_val_adj <0.01) 
write_csv(for_IREA,file = '2024_04_01_Genes_for_IREA.csv')


In [12]:
sessionInfo()

R version 4.3.0 (2023-04-21)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 21.04

Matrix products: default
BLAS/LAPACK: /sc/arion/work/kyriad02/miniconda3/envs/dreamlet430/lib/libopenblasp-r0.3.23.so;  LAPACK version 3.11.0

locale:
[1] C

time zone: America/New_York
tzcode source: system (glibc)

attached base packages:
 [1] grid      parallel  stats4    stats     graphics  grDevices utils    
 [8] datasets  methods   base     

other attached packages:
 [1] metafor_4.2-0               numDeriv_2016.8-1.1        
 [3] metadat_1.2-0               Matrix_1.5-4.1             
 [5] broom_1.0.5                 muscat_1.14.0              
 [7] ComplexHeatmap_2.16.0       circlize_0.4.15            
 [9] aplot_0.1.10                ggtree_3.8.0               
[11] doParallel_1.0.17           iterators_1.0.14           
[13] foreach_1.5.2               scater_1.28.0              
[15] scuttle_1.10.1              SingleCellExperiment_1.22.0
[17] SummarizedExperiment_1.30.2 Ge

In [ ]:
subset_two_cl <- subset(Metacells,subset=TwoClusters %in% c("MTC_123","MTC_456"))
Idents(subset_two_cl) <- "TwoClusters"
DEGs1 <- FindAllMarkers(subset_two_cl,only.pos = T,logfc.threshold = 0,return.thresh = 1)

fold_ch_res <- FoldChange(subset_two_cl,ident.1 = 'MTC_456',
           ident.2 = 'MTC_123',
           logfc.threshold = 0, 
           slot = "data", 
           min.pct = 0.1,
           min.cells.group = 3, 
           pseudocount.use = 1, 
           mean.fxn = NULL, 
           fc.name = NULL, 
           base = 2, 
           return.thresh = 1)

obj <- subset_two_cl@assays$RNA@data[,subset_two_cl$TwoClusters == 'MTC_456']
# Calculate standard deviation of each row
row_sd <- apply(obj, 1, sd)
fold_ch_res$sd.1 <- row_sd

obj <- subset_two_cl@assays$RNA@data[,subset_two_cl$TwoClusters == 'MTC_123']
# Calculate standard deviation of each row
row_sd <- apply(obj, 1, sd)
fold_ch_res$sd.2 <- row_sd

# Calculate Average expressions
averageMT <- as.data.frame(AverageExpression(subset_two_cl,group.by = 'TwoClusters'))


# Merge data frames by row names
merged_df <- merge(DEGs1, fold_ch_res, by = "row.names", all = TRUE)
head(merged_df)
rownames(merged_df) <- merged_df$Row.names
merged_df[,1] <- NULL
merged_df2 <- merge(merged_df,averageMT, by = "row.names", all = TRUE)


rownames(merged_df2) <-  merged_df2$Row.names
merged_df2[,1] <- NULL

table(subset_two_cl$TwoClusters)

se <- sqrt((merged_df2$sd.1^2 / 13313) + (merged_df2$sd.2^2 / 15274))
merged_df2$se <- se

merged_df2$log2FC <- ifelse(merged_df2$cluster=='MTC_456',merged_df2$avg_log2FC.x,-1*merged_df2$avg_log2FC.x)
merged_df2$tstat <-  merged_df2$log2FC /  merged_df2$se
clean_DEGs <- merged_df2 %>% na.omit()
clean_DEGs$gene <- rownames(clean_DEGs)
clean_DEGs <- clean_DEGs%>% arrange(desc(clean_DEGs$log2FC))
readr::write_tsv(clean_DEGs,'2024_03_19_DEGs_MTC_456_vs_123_FULL_nonsig_incl_tstat.tsv')


